In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve
import joblib
import os
import pickle

In [20]:
df = pd.read_csv(r"C:\Users\Mindworx\Desktop\Insurance_Fraud_Detection\Insurance_Claims.csv", sep=';')

In [4]:
for col in ['Monthly_Income', 'Premium_Amount', 'Claim_Amount']:
    df[col] = df[col].astype(str).str.replace(r'[^0-9.]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [5]:
df['Monthly_Income'] = df['Monthly_Income'].fillna(df['Monthly_Income'].median())
df['Premium_Amount'] = df['Premium_Amount'].fillna(df['Premium_Amount'].median())
df['Claim_Amount'] = df['Claim_Amount'].fillna(df['Claim_Amount'].median())

In [6]:
df['Fraud_Flag'] = df['Fraud_Flag'].astype(str).str.strip().str.upper()
df['Fraud_Flag'] = df['Fraud_Flag'].replace({'YES': 1, '1': 1, 'NO': 0, '0': 0, 'N': 0, 'NAN': 0})
df['Fraud_Flag'] = df['Fraud_Flag'].fillna(0).astype(int)

In [7]:
X = df[['Age', 'Monthly_Income', 'Premium_Amount', 'Claim_Amount']]
y = df['Fraud_Flag']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
print("✅ Data Preprocessing Complete!")
print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

✅ Data Preprocessing Complete!
Training samples: 282 | Testing samples: 71


In [11]:
smote = SMOTE(
    sampling_strategy='auto',
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("Before SMOTE")
print(y_train.value_counts())

print("\nAfter SMOTE")
print(y_train_smote.value_counts())

Before SMOTE
Fraud_Flag
0    245
1     37
Name: count, dtype: int64

After SMOTE
Fraud_Flag
1    245
0    245
Name: count, dtype: int64


In [12]:
models = {

    "Logistic Regression":
        LogisticRegression(
            random_state=42,
            max_iter=1000
        ),

    "Decision Tree":
        DecisionTreeClassifier(
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            random_state=42
        )

}

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = []

for name, model in models.items():

    model.fit(X_train_smote, y_train_smote)

    predictions = model.predict(X_test_scaled)

    results.append({

        "Model": name,

        "Accuracy":
            accuracy_score(y_test, predictions),

        "Precision":
            precision_score(y_test, predictions),

        "Recall":
            recall_score(y_test, predictions),

        "F1":
            f1_score(y_test, predictions),
              "ROC-AUC":
            roc_auc_score(y_test, predictions)

    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by="Recall",
    ascending=False
)


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.591549,0.166667,0.555556,0.256410,0.576165
3,Gradient Boosting,0.732394,0.250000,0.555556,0.344828,0.656810
1,Decision Tree,0.661972,0.105263,0.222222,0.142857,0.474014
2,Random Forest,0.704225,0.071429,0.111111,0.086957,0.450717


In [14]:
gb = GradientBoostingClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 5],
    "subsample": [0.8, 1.0]
}

grid = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="recall",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_smote, y_train_smote)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest Recall:")
print(grid.best_score_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Parameters:
{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}

Best Recall:
0.9183673469387754


In [15]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.79      0.83        62
           1       0.13      0.22      0.17         9

    accuracy                           0.72        71
   macro avg       0.50      0.51      0.50        71
weighted avg       0.78      0.72      0.75        71



In [16]:
joblib.dump(best_model, "Models/fraud_model.pkl")


['Models/fraud_model.pkl']

In [17]:
joblib.dump(scaler, "Models/scaler.pkl")

['Models/scaler.pkl']

In [18]:
joblib.dump(X.columns.tolist(), "Models/features.pkl")

['Models/features.pkl']

In [21]:
import os

os.listdir("Models")

['features.pkl', 'fraud_model.pkl', 'scaler.pkl']

In [22]:
print(X.columns.tolist())

['Age', 'Monthly_Income', 'Premium_Amount', 'Claim_Amount']
